### Silver Data transformations

Import all the functions for the transformations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.read.format("delta")\
    .option("header","True")\
    .option("inferSchema","True")\
    .load("abfss://bronze@netflixprojectkiran565.dfs.core.windows.net/netflix_titles")

In [0]:
df.display()

1.Transformation
replacing nulls in duration_minutes and duration_seasons with zeroes

In [0]:
df=df.fillna({"duration_minutes":"0","duration_seasons":"1"})

In [0]:
df.display()

In [0]:
df.printSchema()

### Type cast the first two columns to Integer

In [0]:
df=df.withColumn("duration_minutes",col("duration_minutes").try_cast(IntegerType()))\
    .withColumn("duration_seasons",col("duration_seasons").try_cast(IntegerType()))

In [0]:
df.printSchema()

In [0]:
df.display()

In [0]:
df=df.withColumn("shorttitle",split(col("title"),":")[0])

In [0]:
df.display()

In [0]:
df=df.withColumn("rating",split(col("rating"),"-")[0])
df.display()

In [0]:
df=df.withColumn("Type_flag",when(col("type")=="Movie",1)\
    .when(col("type")=="TV Show",2)\
    .otherwise(0))

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
df=df.withColumn("duration_ranking",dense_rank().over(Window.orderBy(col("duration_minutes").desc())))

In [0]:
df.display()

In [0]:
df.createOrReplaceGlobalTempView("temp_view")

In [0]:
df=spark.sql("select * from temp_view")

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import count
df_vis=df.groupBy("type").agg(count("*").alias("total_count"))


In [0]:
df_vis.display()

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@netflixprojectkiran565.dfs.core.windows.net/netflix_titles").save()